In [35]:
import pandas as pd
import torch
import json
import sys

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from torch import nn, optim

In [36]:
df = pd.read_csv("../dataset/extended_amazon_products.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (180, 2)


,text,category
0,Apple Watch Series 9 GPS smartwatch with fitne...,Electronics_SmartWatch
1,Samsung Galaxy Watch 6 AMOLED smartwatch,Electronics_SmartWatch
2,Noise ColorFit smart watch heart rate monitor,Electronics_SmartWatch
3,Fire-Boltt smart watch bluetooth calling,Electronics_SmartWatch
4,Boat Xtend smart watch with Alexa support,Electronics_SmartWatch


In [37]:
df.tail(4)

,text,category
176,Pet vitamins supplements,Pet_Supplies
177,Cat scratching post,Pet_Supplies
178,Pet toy squeaky,Pet_Supplies
179,Dog harness comfort fit,Pet_Supplies


In [38]:
x = df["text"].values
y = df["category"].values

In [39]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Number of classes:", len(label_encoder.classes_))

# Text → numbers
vectorizer = TfidfVectorizer(max_features=5000)
x_vectorized = vectorizer.fit_transform(x).toarray()

print("Vectorized shape:", x_vectorized.shape)

Number of classes: 9
Vectorized shape: (180, 413)


In [40]:
x_train, x_test, y_train, y_test = train_test_split(
    x_vectorized, y_encoded, test_size=0.2, random_state=42,
    stratify=y_encoded
)

In [41]:
x# Convert to tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)


In [42]:
sys.path.append("..")

from text_model import TextClassifier

model = TextClassifier(
    input_dim=x_train.shape[1],
    num_classes=len(label_encoder.classes_)
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model


TextClassifier(
  (fc1): Linear(in_features=413, out_features=256, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=256, out_features=9, bias=True)
)

In [43]:
epochs=10

model.train()

# Training loop
for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 2.1957
Epoch 2, Loss: 2.1913
Epoch 3, Loss: 2.1841
Epoch 4, Loss: 2.1757
Epoch 5, Loss: 2.1668
Epoch 6, Loss: 2.1614
Epoch 7, Loss: 2.1509
Epoch 8, Loss: 2.1429
Epoch 9, Loss: 2.1350
Epoch 10, Loss: 2.1304


In [44]:
model.eval()

with torch.no_grad():
    test_outputs = model(x_test)
    _, y_pred = torch.max(test_outputs, 1)

y_true = y_test.numpy()
y_pred = y_pred.numpy()

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")

print("\n===== MODEL PERFORMANCE =====")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))



===== MODEL PERFORMANCE =====
Accuracy  : 0.4722
Precision : 0.5893
Recall    : 0.4722
F1 Score  : 0.4406

Classification Report:
                        precision    recall  f1-score   support

Automotive_Accessories       1.00      0.25      0.40         4
       Books_Education       1.00      0.50      0.67         4
         Books_Fiction       0.23      0.75      0.35         4
    Electronics_Camera       0.80      1.00      0.89         4
Electronics_SmartWatch       0.27      0.75      0.40         4
      Fashion_Footwear       0.00      0.00      0.00         4
            Home_Decor       1.00      0.75      0.86         4
          Pet_Supplies       0.00      0.00      0.00         4
        Sports_Fitness       1.00      0.25      0.40         4

              accuracy                           0.47        36
             macro avg       0.59      0.47      0.44        36
          weighted avg       0.59      0.47      0.44        36



c:\New folder\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\New folder\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\New folder\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\New folder\Lib\site-packages\sklearn\metrics\_classification.py:1531: 

In [45]:
metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved to metrics.json")

Metrics saved to metrics.json


In [46]:
torch.save({
    "model_state": model.state_dict(),
    "vectorizer": vectorizer,
    "label_encoder": label_encoder
}, "artifacts.pth")

print("Model artifacts saved to artifacts.pth")

Model artifacts saved to artifacts.pth
